# RP-1/LOX design-study workflow

This notebook reads RPA4 CFG/PDF pairs from `RPA4_SHARED_DIR` and plots Mach values directly from each report. It also demonstrates bounded candidate sampling, ideal CEA screening, optional explicitly configured Cantera screening, manual RPA4 batch staging, and multi-objective ranking.

RPA4 still runs on the other computer. This project writes CFG candidates and imports completed PDFs; it does not automate RPA4. The example parameter ranges are placeholders around the supplied case, not approved hot-fire operating limits. Set actual requirements and hard constraints before selecting hardware. The nozzle inlet area-ratio discrepancy is a review warning because its PDF-to-CFG station mapping is not established.

In [ ]:
import os
from pathlib import Path

from rocket_analysis.plotting import plot_mach_reference
from rocket_analysis.rpa4 import mach_stations_from_report_values
from rocket_analysis.shared import load_shared_rpa4_cases

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
shared_dir = os.environ.get("RPA4_SHARED_DIR")
active_folder = Path(shared_dir).expanduser() if shared_dir else ROOT / "data" / "reference"
shared_cases = load_shared_rpa4_cases(active_folder, allow_incomplete=True)

baseline_name = os.environ.get("RPA4_BASELINE_CASE")
if baseline_name:
    selected_case = next(
        (case for case in shared_cases if case.case_name.casefold() == baseline_name.strip().casefold()),
        None,
    )
    if selected_case is None:
        raise ValueError(f"RPA4_BASELINE_CASE={baseline_name!r} is not present in {active_folder}")
else:
    selected_case = shared_cases[0]

rpa4_inputs = selected_case.inputs
report_values = selected_case.report_values
mach_stations = mach_stations_from_report_values(report_values)
print("Source folder:", active_folder)
print("Selected case:", selected_case.case_name)
print("CFG:", selected_case.config_path)
print("PDF:", selected_case.report_path)
print("Validation:", selected_case.validation["status"], selected_case.validation["counts"])

In [ ]:
print("O/F:", rpa4_inputs.mixture_ratio)
print("Chamber pressure:", rpa4_inputs.chamber_pressure)
print("Throat diameter:", rpa4_inputs.throat_diameter)
print("RPA4 Table 2 Mach samples:", [(point.name, point.mach) for point in mach_stations])
print("Estimated-delivered performance:", report_values["estimated_delivered"])

In [ ]:
[check for check in selected_case.validation["checks"] if check["status"] != "match"]

In [ ]:
plot_mach_reference(
    mach_stations,
    title=f"RPA4 Table 2 Mach by station ? {selected_case.case_name}",
)

In [ ]:
from rocket_analysis.ranking import candidate_metric_catalog, rank_metric_rows

report_candidates = [
    {
        "candidate_id": case.case_name,
        "case_name": case.case_name,
        "validation_status": case.validation["status"],
        "metrics": candidate_metric_catalog(case.report_values),
    }
    for case in shared_cases
]
report_ranking = rank_metric_rows(report_candidates, {
    "mode": "pareto",
    "objectives": [
        {"metric": "estimated_delivered.sea_level.thrust_kN", "direction": "maximize"},
        {"metric": "estimated_delivered.sea_level.specific_impulse_s", "direction": "maximize"},
    ],
    "constraints": [],  # Add measured design and test limits before comparing hot-fire candidates.
})
print("Ranking mode:", report_ranking.mode)
print("Top Pareto candidates:", report_ranking.top_ranked)
weighted_ranking = rank_metric_rows(report_candidates, {
    "mode": "weighted",
    "objectives": [
        {"metric": "estimated_delivered.sea_level.thrust_kN", "direction": "maximize"},
        {"metric": "estimated_delivered.sea_level.specific_impulse_s", "direction": "maximize"},
    ],
    "weights": {
        "estimated_delivered.sea_level.thrust_kN": 0.5,
        "estimated_delivered.sea_level.specific_impulse_s": 0.5,
    },
})
print("Weighted-score order:", weighted_ranking.entries)
print("All report metrics are retained per candidate for additional objectives:", len(report_candidates[0]["metrics"]))
print("Excluded candidates:", report_ranking.exclusions)

## Build a range study

The example JSON spec varies O/F, chamber pressure, throat diameter, nozzle exit area ratio, and the RPA4 nozzle inlet area-ratio field. It samples 32 Sobol points. Its limits are demonstration values around the transferred case; replace them with engineering requirements and add hard constraints for the actual test article. Grid and Optuna adaptive sampling are also supported.

NASA CEA is used for ideal equilibrium screening only. Cantera remains disabled until a mechanism and explicit RP-1 surrogate species mapping are supplied. After screening, staged CFG files must be run manually in RPA4 on the other computer.

In [ ]:
from rocket_analysis.study import (
    generate_candidate_cfgs,
    import_shared_rpa4_results,
    load_study_spec,
    rank_screened_candidates,
    rank_study_results,
    sample_parameter_sets,
    save_study_ranking,
    screen_study_candidates,
    stage_rpa4_candidates,
)

study_spec = load_study_spec(ROOT / "data" / "studies" / "rp1-lox-design-study.example.json")
preview, _ = sample_parameter_sets(study_spec)
print("Sampling method:", study_spec.sampling_method)
print("Candidate count:", len(preview))
print("First candidate values:", preview[0])

# Set GENERATE_RPA4_BATCH=1 before launching Jupyter to write candidates, screen, and stage CFGs.
if os.environ.get("GENERATE_RPA4_BATCH") == "1":
    generated = generate_candidate_cfgs(study_spec)
    screening = screen_study_candidates(study_spec, generated)
    cea_ranking = rank_screened_candidates(study_spec, {
        "mode": "pareto",
        "objectives": [
            {"metric": "screening.cea.result.ideal_vacuum_specific_impulse_m_s", "direction": "maximize"},
            {"metric": "screening.cea.result.characteristic_velocity_m_s", "direction": "maximize"},
        ],
    })
    print("CEA screen ranking:", cea_ranking.entries[:10])
    staged = stage_rpa4_candidates(study_spec, candidates=generated)
    print("CFGs staged for manual RPA4 runs:", len(staged))

# After the other computer writes the PDFs into those same case folders, set IMPORT_RPA4_RESULTS=1.
if os.environ.get("IMPORT_RPA4_RESULTS") == "1":
    imported = import_shared_rpa4_results(study_spec)
    final_ranking = rank_study_results(study_spec)
    ranking_path = save_study_ranking(study_spec, final_ranking)
    print("Imported RPA4 candidates:", imported["last_imported_candidate_ids"])
    print("Ranking JSON:", ranking_path)
    print("Top candidates:", final_ranking.top_ranked)
    print("Excluded candidates:", final_ranking.exclusions)

## Ranking and validation scope

Pareto ranking keeps candidates that trade delivered sea-level thrust against delivered sea-level Isp. Weighted ranking is a separate option; it scales each selected metric from 0 to 1 across the eligible candidates in the current import, then applies the configured weights. Report metrics are retained with unit-bearing names, so objectives and constraints can include any parsed performance, station, geometry, mass-flow, propellant, or altitude-sweep value. Review findings such as the nozzle inlet mapping warning remain eligible; hard CFG/PDF mismatches and failed constraints are excluded.

The CEA cross-check agrees closely with the supplied ideal c-star and exit Mach. Cantera returns only equilibrium chamber-state properties until a suitable mechanism and fuel mapping are supplied. OpenFOAM hooks require a user-provided, reviewed case template and should be used only for finalists. None of these calculations establishes measured hot-fire performance or material/cooling/structural suitability.